# Inspecting 4D-STEM .hspy data

This notebook inspects .hspy data, acquired at JEOL Grand ARM TEM.

## Contents
 1. [Load and inspect the data](#1.Load-and-inspect-the-data)

In [ ]:
%matplotlib widget

# import warnings
# warnings.filterwarnings('ignore')

import hyperspy.api as hs #General hyperspy package
import pyxem as pxm #Electron diffraction tools based on hyperspy
import numpy as np #General numerical and matrix support
import matplotlib.pyplot as plt #Plotting tools
import matplotlib.colors as mcolors #Some plotting color tools
from matplotlib.cm import ScalarMappable
import diffpy #Electron diffraction tools
import requests
import dask.array as da
import ipympl
import ipywidgets


import pandas as pd
import seaborn as sb
from tabulate import tabulate

#Import path handling tool
from pathlib import Path
import os



# 1. Load and inspect the data

In [ ]:
base_directory = Path("/Users/ingridmzg/Masteroppgave/Bilder/GrandARM/transfer_437934_files_e5e30d0b/")
files = list(base_directory.glob("*.hspy"))
print("Found files:")
for i, f in enumerate(files):
    print(f'{i}:/t{f}')

In [ ]:
# Select indices from the printed list
signal_idx = 14
navigator_idx = 0

signal_path = files[signal_idx]
navigator_path = files[navigator_idx]

In [ ]:
# load lazy

lazy = True
signal = hs.load(str(signal_path), lazy=lazy) #Load the data. Use `lazy=True` if you are experiencing memory issues
if lazy:
    signal.rechunk(nav_chunks=(32, 32),sig_chunks=(32, 32))

pnADF = hs.load(str(navigator_path), lazy=True)

In [ ]:
#datapath = Path("/Users/ingridmzg/Masteroppgave/Bilder/GrandARM/transfer_437934_files_e5e30d0b/0001_4D-STEM_Single_pnADF_SED128x128.hspy")

The dimensions in hyperspy are defined as $(X, Y|k_x, k_y)$, where $X$ and $Y$ are the real-space (also called "navigation axes", and $k_x$ and $k_y$ are the reciprocal axes (also called the "signal axes").

In [ ]:
signal

In [ ]:
signal.axes_manager

In [ ]:
print(f'The signal:\n{signal}\n') #Print information about the signal
print(f'Axes information:\n{signal.axes_manager}\n') #Print information about the axes
print(f'Data:\n{signal.data}\n')

### Metadata

In [ ]:
print(f'The signal metadata:\n{signal.metadata}')
print(f'The signal axes:\n{signal.axes_manager}')

# 2. Reshape

In [ ]:
#signal = pxm.signals.LazyElectronDiffraction2D(signal)


In [ ]:
signal.set_signal_type('electron_diffraction')

In [ ]:
print(f'The signal:\n{signal}\n') #Print information about the signal
print(f'Axes information:\n{signal.axes_manager}\n') #Print information about the axes
print(f'Data:\n{signal.data}\n')

# Plotting

In [ ]:
signal.plot()

In [ ]:
pnADF.plot()
#signal.plot()
signal.plot(norm='symlog', navigator=pnADF)

In [ ]:
# create an averaged maximum image
dp_max = signal.max(axis=[0, 1])
dp_max.plot(norm='symlog', cmap="magma", axes_off=True)

# 6. Crop

In [ ]:
pnADF.compute()

In [ ]:
pnADF.plot()
left, right, top, bottom = pnADF.axes_manager.signal_extent #Set the roi to cover the whole scan area
left, right, top, bottom = 100.081, 80.572, 6.630999999999999, 17.799 #Select a pre-defined region
roi = hs.roi.RectangularROI(left=left, right=right, top=top, bottom=bottom)
roi.add_widget(pnADF)

In [ ]:
print(roi.left, roi.right, roi.top, roi.bottom)

In [ ]:
pnADF = roi(pnADF)

In [ ]:
signal = roi(signal) 


In [ ]:
print(pnADF)
print(signal)

In [ ]:
pnADF.plot()
#signal.plot()


In [ ]:
signal.plot(norm='symlog', navigator=pnADF.T)

In [ ]:
assert False

In [ ]:
pnADF.axes_manager[0].convert_units()

In [ ]:
sig_crop = False
nav_crop = True

if sig_crop:
    s = signal.isig[128:512-128, 128:512-128].deepcopy()
elif nav_crop:
    s = signal.inav[::8, ::8].deepcopy()
else:
    s = signal

In [ ]:
s

In [ ]:
signal.plot()
left, right, top, bottom = signal.axes_manager.signal_extent #Set the roi to cover the whole scan area
left, right, top, bottom = 150.081, 80.572, 6.630999999999999, 17.799 #Select a pre-defined region
roi = hs.roi.RectangularROI(left=left, right=right, top=top, bottom=bottom)
roi.add_widget(signal)

In [ ]:
print(roi.left, roi.right, roi.top, roi.bottom)

In [ ]:
# crop the data. 
signal = roi(signal) 


In [ ]:
# plot the cropped data
signal.plot(norm='symlog')

In [ ]:
# save the cropped data

signal.save(str(signal_path.parent / "cropped_signal.hspy"))

# 8. Calibrate diffraction space

In [ ]:
signal = hs.load(signal.append(f'cropped', 'hspy'), lazy=True)
signal.set_diffraction_calibration(1.0)

In [ ]:
signal.plot(norm='symlog')

In [ ]:
assert False

In [ ]:
signal.plot(norm='symlog')
point = hs.roi.Point2DROI(0,0)
pattern = point.interactive(signal, axes=[0, 1])
pattern.plot(norm='symlog')
roi = hs.roi.Line2DROI(-0.05, -0.05, 0.05, 0.05, linewidth=0.001)
profile = roi.interactive(pattern)
profile.plot(norm='log')
span = hs.roi.SpanROI()
span.add_widget(profile)
profile = roi.interactive(signal, axes=[-2, -1])
profile.plot(norm='log')

In [ ]:
profile.axes_manager[0].scale

In [ ]:
profile.axes_manager[-1].units

In [ ]:
signal.metadata.Acquisition_instrument.TEM.beam_energy

In [ ]:
r = (span.right-span.left)
old_scale = profile.axes_manager[-1].scale
units = pattern.axes_manager[-1].units
h, k, l = 1, 1, 1
n = 3
hkl = np.array([h, k, l])
a = 5.653 #Å
g = np.sqrt(np.sum((hkl/a)**2))
print(f'Calculated g-vector: {g:.3e} Å^-1')

if units =='rad':
    wavelength=pxm.utils.calibration.get_electron_wavelength(s.metadata.Acquisition_instrument.TEM.beam_energy)
    g = g * wavelength #convert to radians

scale = g/r

print(f'Measured 3x({h}{k}{l}) distance: {r:.3e} {units}\nMeasured distance in pixels: {r/old_scale} [px]\nCalculated g-vector: {g:.3e} Å^-1\nOld scale: {old_scale:.3e} {units}/px\nCalculated scale: {scale:.3e} {units}/px')

In [ ]:
a_GaAs = 5.653 #Å
h, k, l = 1, 1, 1
n = 6
hkl = np.array([h, k, l])*n
g_hkl = np.sqrt(np.sum((hkl/a_GaAs)**2))
L = (span.right-span.left) / profile.axes_manager[0].scale
scale = g_hkl / L
print(f'Lattice spacing of ({h*n}, {k*n}, {l*n}) is {g_hkl:.3g} 1/Å\nMeasured distance on detector is {L:.2f}\nDiffraction scale is {scale:.3g} 1/Å')

In [ ]:
signal.set_diffraction_calibration(scale)

In [ ]:
signal.save(str(signal_path.parent / "calibrated_signal.hspy"))